# P04 · Notebook 34 · The 8-microphone array with the four simple families

**Why.** Notebook 11 showed that aiming MIMII's eight microphones at the pump helps the pre-trained model in factory noise. Families 1 to 4 were only run on one microphone, at the quietest level. This notebook runs them on all three front ends, at all three noise levels.

**What it does, for the four MIMII pumps at 6, 0 and −6 dB:**
- three front ends, with notebook 11's code: one microphone (CH0), the plain average of eight (AVG), and the eight aimed at the pump (BF, delays measured once at 6 dB);
- the four families, with the classes and settings of the pump runs, unchanged: a limit on loudness, distance from the normal cloud, Isolation Forest, the autoencoder;
- the same split as the pump runs (Split A), so one microphone at 6 dB must reproduce them (gate G0b).

**Only MIMII has eight microphones.** The DCASE recordings are one channel, so the array cannot be tried on the three public tests.

**Rules** written before running: `results/mimii_beats/RULES_array_simple_families_written_before_running.md`.

**Run:**
- Runtime › Run all. **No GPU is needed.** Choose a CPU runtime with high RAM if you have it.
- It downloads three zips of about 8 GB each from Zenodo (one at a time; each is deleted after use).
- **Time, estimated:** about 1.5 hours for the sound and families 1 to 3, then the autoencoder: 36 networks. In notebook 31 one such network took 7.9 minutes on average, so about 5 hours. Families 1 to 3 are printed before the autoencoders start.
- Everything is saved to Drive › `P04_ArraySimple` as it finishes (about 1.5 GB of log-mel pictures, which you can delete afterwards). If Colab disconnects, run all again: finished levels and finished networks are skipped.

**Licence:** MIMII: CC BY-SA 4.0 (Hitachi).

In [ ]:
# 1 · Setup (no GPU is needed)
import os, sys, json, time, hashlib, struct, zipfile, subprocess, shutil, urllib.request, re
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
import sklearn

TEST = os.environ.get("P04_TEST") == "1"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT, WORK = Path("/content/drive/MyDrive/P04_ArraySimple"), Path("/content/work")
else:
    OUT, WORK = Path(os.environ.get("P04_OUT", "P04_ArraySimple")), Path(os.environ.get("P04_WORK", "work"))
for p in (OUT, WORK, OUT / "scores"):
    p.mkdir(parents=True, exist_ok=True)
POOL = ThreadPoolExecutor(max(2, os.cpu_count() or 2))
print("CPU threads:", os.cpu_count(), "· numpy", np.__version__, "· scikit-learn", sklearn.__version__)


def md5(path, chunk=8 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def download(url, dest):
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    try:
        if url.startswith("file://"):
            shutil.copy(url[7:], part)
        elif shutil.which("wget"):
            subprocess.run(["wget", "-q", "--show-progress", "-c", "-O", str(part), url], check=True)
        else:
            urllib.request.urlretrieve(url, part)
        part.rename(dest)
    finally:
        if part.exists() and not dest.exists():
            part.unlink()

In [ ]:
# 2 · The files, and what was fixed in the rules before this notebook ran
RULES = "results/mimii_beats/RULES_array_simple_families_written_before_running.md"
LEVELS = {"6dB": ("https://zenodo.org/records/3384388/files/6_dB_pump.zip?download=1", "6_dB_pump.zip", "a09ba6060c10fc09cd4c8770213b0b9f"),
          "0dB": ("https://zenodo.org/records/3384388/files/0_dB_pump.zip?download=1", "0_dB_pump.zip", "488748295c3f60b25de07b58fe75b049"),
          "-6dB": ("https://zenodo.org/records/3384388/files/-6_dB_pump.zip?download=1", "-6_dB_pump.zip", "d20b783a0ff9c93d58f452f98c37b112")}
SR, NS = 16000, 160000
if TEST:
    LEVELS = json.loads(os.environ["P04_TEST_DATA"]); NS = int(json.loads(os.environ.get("P04_TEST_CFG", "{}")).get("clip_samples", NS))
FRONTS = ["CH0", "AVG", "BF"]                        # one microphone · plain average of eight · eight aimed at the pump (delays from 6 dB)
FRONT_NAME = {"CH0": "one microphone", "AVG": "average of eight, not aimed", "BF": "eight, aimed at the pump"}
NFFT, HOP, UPS, MAXLAG = 1024, 512, 16, 16          # GCC-PHAT, as notebook 11: frame, hop, zero-padding factor, search ±16 samples
FIRST = list(LEVELS)[0]                              # 6 dB: the installation measurement
FAMS = ["F1", "F2", "F3", "F4"]
FAM_NAME = {"F1": "Family 1 · a limit on one number (loudness)", "F2": "Family 2 · distance from the normal cloud (128 clip statistics)",
            "F3": "Family 3 · isolation (Isolation Forest, 128 clip statistics)", "F4": "Family 4 · autoencoder (reconstruction error)",
            "F1_raw": "reported beside · loudness as it is (louder = more anomalous)"}
SCORED = FAMS + ["F1_raw"]
N_FFT, N_MELS = 1024, 64                             # the log-mel picture of the pump runs
BAR_NOISE, BAR_Q1, BAR_Q2 = 0.02, 0.02, 0.01         # the bars of the three questions (rules)
# records the gates compare with
N11_TAU_6DB = {"id_00": [0.0, -0.9375, -1.125, -0.0625, 1.0, 4.0, 2.75, 1.125], "id_02": [0.0, -0.8125, -1.875, -0.8125, 0.1875, 4.0, 2.6875, 1.0], "id_04": [0.0, -1.0, -1.5, -1.125, -0.1875, 0.9375, 2.5, 1.125], "id_06": [0.0, -0.9375, -1.6875, -1.625, -0.6875, 3.3125, 2.125, 0.5625]}
N11_FAMILY5_FROZEN = {"6dB": {"CH0": 0.9904, "AVG": 0.9898, "BF": 0.9909}, "0dB": {"CH0": 0.9525, "AVG": 0.9616, "BF": 0.9739}, "-6dB": {"CH0": 0.8234, "AVG": 0.8495, "BF": 0.8795}}      # notebook 11: the pre-trained model alone, Split A, average of 4 pumps
PUMP_RECORD = {"F2": {"id_00": 0.9997, "id_02": 0.8719, "id_04": 0.9945, "id_06": 0.9995}, "F3": {"id_00": 0.9614, "id_02": 0.9602, "id_04": 0.9642, "id_06": 0.9237}, "F4": {"id_00": 0.8254, "id_02": 0.4578, "id_04": 0.9833, "id_06": 0.9069}, "F1_raw": {"id_00": 0.6643, "id_02": 0.7919, "id_04": 0.8061, "id_06": 0.6806}}      # results/mimii: one microphone, 6 dB, Split A


def get_zip(level):
    url, name, want = LEVELS[level]
    for place in (OUT / name, WORK / name):
        if place.exists() and md5(place) == want:
            return place
    dest = WORK / name
    t = time.time(); download(url, dest)
    got = md5(dest)
    print(f"{name}: downloaded in {time.time() - t:.0f} s · MD5 matches Zenodo: {got == want}")
    assert got == want, f"stop: {name} is not the published file"
    return dest


def read_all(data):
    # WAV bytes (plain or WAVE_FORMAT_EXTENSIBLE, 16-bit PCM) -> (samples, channels) float32 in [-1, 1], sample rate
    assert data[:4] == b"RIFF" and data[8:12] == b"WAVE", "not a WAV file"
    pos, fmt, raw = 12, None, None
    while pos + 8 <= len(data):
        cid, size = data[pos:pos + 4], struct.unpack("<I", data[pos + 4:pos + 8])[0]
        body = data[pos + 8:pos + 8 + size]
        if cid == b"fmt ":
            tag, ch, sr, _, _, bits = struct.unpack("<HHIIHH", body[:16])
            if tag == 0xFFFE:
                tag = struct.unpack("<H", body[24:26])[0]
            fmt = (tag, ch, sr, bits)
        elif cid == b"data":
            raw = body
        pos += 8 + size + (size & 1)
    tag, ch, sr, bits = fmt
    assert tag == 1 and bits == 16, f"expected 16-bit PCM, got format {tag}, {bits}-bit"
    nf = len(raw) // (2 * ch)
    return np.frombuffer(raw[:nf * 2 * ch], dtype="<i2").reshape(-1, ch).astype(np.float32) / 32768.0, sr


def parse(names):
    pump = np.array([re.search(r"(id_\d+)", n).group(1) for n in names])
    label = np.array([1 if n.replace("\\", "/").split("/")[-2] == "abnormal" else 0 for n in names])
    return pump, label


def split_a(pump, label, fname, p):
    # the same split as notebooks 1 and 3: test = first N normal by name + all abnormal of pump p; train = the other normal
    idx = np.where(pump == p)[0]
    nor = idx[label[idx] == 0]; abn = idx[label[idx] == 1]
    nor = nor[np.argsort(fname[nor])]
    return nor[len(abn):], np.r_[nor[:len(abn)], abn]

In [ ]:
# 3 · The array physics of notebook 11, unchanged: GCC-PHAT delays from training clips, then delay-and-sum on the whole clip
GCC_WIN = np.hanning(NFFT).astype(np.float32)


def cross_spectra(x):
    # x: (samples, 8). Sum over frames of conj(X_0)·X_m for m = 0..7 → (8, NFFT/2+1) complex
    n = (len(x) - NFFT) // HOP + 1
    idx = np.arange(NFFT)[None, :] + HOP * np.arange(n)[:, None]
    F = np.fft.rfft(x[idx] * GCC_WIN[None, :, None], axis=1)           # (frames, bins, 8)
    return np.einsum("fb,fbm->mb", np.conj(F[:, :, 0]), F)


def delays_from(G):
    # GCC-PHAT on summed cross-spectra G (8, bins) → delay of each channel against channel 0, in samples
    out = np.zeros(G.shape[0])
    for m in range(1, G.shape[0]):
        g = G[m] / np.maximum(np.abs(G[m]), 1e-12)
        r = np.fft.irfft(g, n=NFFT * UPS)
        K = MAXLAG * UPS
        rc = np.concatenate([r[-K:], r[:K + 1]])                     # lags −K..+K (in 1/UPS samples)
        out[m] = (np.argmax(rc) - K) / UPS
    return out


def delay_and_sum(X, tau, N):
    k = np.arange(X.shape[0])[:, None]
    return np.fft.irfft((X * np.exp(2j * np.pi * k * tau[None, :] / N)).mean(1), n=N).astype(np.float32)


def front_ends(x, tau, tau_same=None):
    # x: (samples, 8) → mono signals; BF advances channel m by tau[m] samples, then averages
    N = len(x)
    X = np.fft.rfft(x, axis=0)
    out = {"CH0": x[:, 0].copy(), "AVG": x.mean(1).astype(np.float32), "BF": delay_and_sum(X, tau, N)}
    if tau_same is not None:
        out["BFs"] = delay_and_sum(X, tau_same, N)
    return out


# smoke test: a broadband source arriving with known fractional delays, plus independent noise 6 dB louder
rng = np.random.default_rng(0)
true = np.r_[0.0, rng.uniform(-4, 4, 7)]
s = rng.standard_normal(160000).astype(np.float32)
S = np.fft.rfft(s); kk = np.arange(len(S))
xs = np.stack([np.fft.irfft(S * np.exp(-2j * np.pi * kk * t / 160000), n=160000) for t in true], 1).astype(np.float32)
xn = xs + 2.0 * rng.standard_normal(xs.shape).astype(np.float32)
est = delays_from(cross_spectra(xn))
fe = front_ends(xn, est)
gain = lambda y: 10 * np.log10(np.var(s) / np.var(y - s * (np.dot(y, s) / np.dot(s, s))))
print("delays true", np.round(true, 2).tolist(), "\nestimated  ", np.round(est, 2).tolist())
print(f"SNR after each front end (dB): CH0 {gain(fe['CH0']):.1f} · AVG {gain(fe['AVG']):.1f} · BF {gain(fe['BF']):.1f}")
assert np.max(np.abs(est - true)) < 0.25, "smoke test: GCC-PHAT does not recover the delays"
assert gain(fe["BF"]) > gain(fe["CH0"]) + 6, "smoke test: delay-and-sum does not gain the expected ~9 dB"

In [ ]:
# 4 · For each noise level: the three front ends of every clip, then its log-mel picture (code/p04_mimii_features.py, unchanged:
# 64 bands, slices of 1,024 readings, a new slice every 512). Saved to Drive per level; a finished level is not redone
EPS = sys.float_info.epsilon


def hz_to_mel(f):
    """Slaney mel scale (librosa htk=False)."""
    f = np.asanyarray(f, dtype=float)
    f_sp = 200.0 / 3; mels = f / f_sp
    min_log_hz = 1000.0; min_log_mel = min_log_hz / f_sp; logstep = np.log(6.4) / 27.0
    return np.where(f >= min_log_hz, min_log_mel + np.log(np.maximum(f, 1e-10) / min_log_hz) / logstep, mels)


def mel_to_hz(m):
    m = np.asanyarray(m, dtype=float)
    f_sp = 200.0 / 3; freqs = f_sp * m
    min_log_hz = 1000.0; min_log_mel = min_log_hz / f_sp; logstep = np.log(6.4) / 27.0
    return np.where(m >= min_log_mel, min_log_hz * np.exp(logstep * (m - min_log_mel)), freqs)


def mel_filters(sr=SR, n_fft=N_FFT, n_mels=N_MELS):
    """Triangular filters with Slaney area normalisation (librosa.filters.mel defaults)."""
    fftfreqs = np.fft.rfftfreq(n_fft, 1.0 / sr)
    mel_f = mel_to_hz(np.linspace(hz_to_mel(0.0), hz_to_mel(sr / 2.0), n_mels + 2))
    fdiff = np.diff(mel_f); ramps = mel_f[:, None] - fftfreqs[None, :]
    w = np.zeros((n_mels, len(fftfreqs)))
    for i in range(n_mels):
        lower = -ramps[i] / fdiff[i]; upper = ramps[i + 2] / fdiff[i + 1]
        w[i] = np.maximum(0, np.minimum(lower, upper))
    w *= (2.0 / (mel_f[2:n_mels + 2] - mel_f[:n_mels]))[:, None]
    return w


def power_spec(y):
    """|STFT|² with centred frames and zero padding (librosa ≥ 0.10 default pad_mode='constant')."""
    y = np.pad(y, (N_FFT // 2, N_FFT // 2), mode="constant")
    n = 1 + (len(y) - N_FFT) // HOP
    idx = np.arange(N_FFT)[None, :] + HOP * np.arange(n)[:, None]
    return (np.abs(np.fft.rfft(y[idx] * WIN, axis=1)) ** 2).T


MEL = mel_filters(SR, N_FFT, N_MELS)
WIN = 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(N_FFT) / N_FFT)       # periodic Hann, as in the pump runs
N_FRAMES = 1 + NS // HOP
BATCH = 24


def batches(seq, n=BATCH):
    for i in range(0, len(seq), n):
        yield seq[i:i + n]


def read8(data):
    x, sr = read_all(data)
    assert sr == SR and x.shape == (NS, 8), (sr, x.shape)
    return x


def pictures(args):
    data, tau = args
    fe = front_ends(read8(data), tau)
    return [(10.0 * np.log10(MEL @ power_spec(fe[f]) + EPS)).astype(np.float16) for f in FRONTS]


def lm_file(level, f, where=None):
    return (where or OUT) / f"lm_{level}_{f}.npy"


def level_ready(level):
    return (OUT / f"meta_{level}.npz").exists() and all(lm_file(level, f).exists() for f in FRONTS)


def run_level(level):
    if level_ready(level):
        print(level, "already done"); return
    zpath = get_zip(level); zf = zipfile.ZipFile(zpath)
    names = np.array(sorted(n for n in zf.namelist() if n.lower().endswith(".wav")))
    pump, label = parse(names); pumps = sorted(set(pump.tolist())); t0 = time.time()
    dfile = OUT / f"delays_{FIRST}.json"
    if level == FIRST and not dfile.exists():                        # pass 1: the installation measurement, on training clips only
        tau, half = {}, {}
        for p in pumps:
            tr, _ = split_a(pump, label, names, p); tr = tr[np.argsort(names[tr])]
            Gs = [0, 0]
            for chunk in batches(list(enumerate(tr))):
                G = list(POOL.map(lambda d: cross_spectra(read8(d)), [zf.read(names[i]) for _, i in chunk]))
                for (j, _), g in zip(chunk, G):
                    Gs[j % 2] = Gs[j % 2] + g
            tau[p] = delays_from(Gs[0] + Gs[1]).tolist(); half[p] = [delays_from(Gs[0]).tolist(), delays_from(Gs[1]).tolist()]
            print(f"  {level} {p}: delays (samples) {np.round(tau[p], 2).tolist()} · {time.time() - t0:.0f} s", flush=True)
        dfile.write_text(json.dumps({"tau": tau, "half": half}))
    tau = {p: np.array(v) for p, v in json.loads(dfile.read_text())["tau"].items()}
    tmp = {f: lm_file(level, f, WORK).with_name(f"lm_{level}_{f}.part.npy") for f in FRONTS}
    mm = {f: np.lib.format.open_memmap(tmp[f], mode="w+", dtype=np.float16, shape=(len(names), N_MELS, N_FRAMES)) for f in FRONTS}
    k = 0
    for chunk in batches(list(range(len(names)))):                   # pass 2: every clip
        for pics in POOL.map(pictures, [(zf.read(names[i]), tau[pump[i]]) for i in chunk]):
            for f, lm in zip(FRONTS, pics):
                mm[f][k] = lm
            k += 1
        if (k // BATCH) % 20 == 0:
            print(f"  {level}: {k}/{len(names)} clips · {(time.time() - t0) / 60:.1f} min", flush=True)
    for f in FRONTS:
        mm[f].flush()
    del mm
    for f in FRONTS:                                                 # local file first, then one copy to Drive
        os.replace(tmp[f], lm_file(level, f, WORK))
        part = lm_file(level, f).with_name(f"lm_{level}_{f}.part"); shutil.copy(lm_file(level, f, WORK), part); os.replace(part, lm_file(level, f))
    np.savez(OUT / f"meta_{level}.part.npz", fname=names, pump=pump, label=label, zip_md5=LEVELS[level][2], seconds=round(time.time() - t0, 1))
    os.replace(OUT / f"meta_{level}.part.npz", OUT / f"meta_{level}.npz")
    print(f"{level}: {len(names)} clips, three front ends → saved · {(time.time() - t0) / 60:.1f} min")
    if IN_COLAB and zpath.parent == WORK:
        zpath.unlink()                                               # free the disk for the next level


for level in LEVELS:
    run_level(level)
META = {lv: dict(np.load(OUT / f"meta_{lv}.npz", allow_pickle=False)) for lv in LEVELS}
PUMPS = sorted(set(META[FIRST]["pump"].tolist()))
assert all(np.array_equal(META[lv]["pump"], META[FIRST]["pump"]) and np.array_equal(META[lv]["label"], META[FIRST]["label"]) for lv in LEVELS)
print("pumps:", PUMPS, "· clips per level:", len(META[FIRST]["pump"]))

In [ ]:
# 5 · The four families (code/p04_mimii_models.py, unchanged classes). A unit is one noise level, one front end and one pump:
# each family is fitted on that pump's normal training clips (Split A of the pump runs) and scores that pump's test clips
def frames5(lm):
    """(64, T) log-mel → (T-4, 320): five consecutive frames side by side, as the baseline."""
    T = lm.shape[1]
    return np.concatenate([lm[:, t:T - 4 + t].T for t in range(5)], axis=1)


def clip_stats(logmel):
    x = logmel.astype(np.float32)
    return np.c_[x.mean(2), x.std(2)]


class AEfam:
    """M1, set up like the Keras baseline: no L2 penalty (Keras adds none), and the last 10 % of the
    training rows left out (Keras validation_split=0.1 holds them out before shuffling).
    Chosen on 2026-11-06 after the inspection (results/mimii/DECISION_after_inspection.md). It is also
    about five times faster: scikit-learn's default L2 slowly pushes dead-unit weights into float32
    denormals. The first run (alpha 0.0001, all rows) is kept in results/mimii/first_run/."""
    ALPHA, KEEP = 0.0, 0.9

    def fit(self, LM):
        X = np.concatenate([frames5(m.astype(np.float32)) for m in LM])
        X = X[: int(len(X) * self.KEEP)]                  # Keras: split_at = int(len(x) * (1 - validation_split))
        self.m = MLPRegressor(hidden_layer_sizes=(64, 64, 8, 64, 64), activation="relu", solver="adam", alpha=self.ALPHA,
                              batch_size=512, max_iter=50, n_iter_no_change=50, tol=0, shuffle=True,
                              random_state=0).fit(X, X)
        return self

    def score(self, LM):
        out = []
        for m in LM:
            X = frames5(m.astype(np.float32))
            out.append(float(np.mean((self.m.predict(X) - X) ** 2)))
        return np.array(out)


class Maha:
    """Standardise → PCA (keep 95 % or all) → T² + SPE, each over its 99th training percentile."""
    def __init__(self, var=0.95):
        self.var = var

    def fit(self, X):
        self.sc = StandardScaler().fit(X); Z = self.sc.transform(X)
        p = PCA().fit(Z)
        k = len(p.explained_variance_) if self.var is None else int(np.searchsorted(np.cumsum(p.explained_variance_ratio_), self.var) + 1)
        self.P, self.lam = p.components_[:k].T, p.explained_variance_[:k]
        t2, spe = self._parts(Z)
        self.qt, self.qs = np.quantile(t2, 0.99), max(np.quantile(spe, 0.99), 1e-12)
        return self

    def _parts(self, Z):
        T = Z @ self.P
        return (T ** 2 / self.lam).sum(1), ((Z - T @ self.P.T) ** 2).sum(1)

    def score(self, X):
        t2, spe = self._parts(self.sc.transform(X))
        return t2 / self.qt + spe / self.qs


class IFfam:
    def fit(self, X):
        self.sc = StandardScaler().fit(X)
        self.m = IsolationForest(n_estimators=200, random_state=0).fit(self.sc.transform(X))
        return self

    def score(self, X):
        return -self.m.score_samples(self.sc.transform(X))


_LM = {}


def LM(level, f):
    if (level, f) not in _LM:
        local = lm_file(level, f, WORK)
        if not local.exists():
            shutil.copy(lm_file(level, f), local.with_suffix(".part")); os.replace(local.with_suffix(".part"), local)
        _LM[(level, f)] = np.load(local, mmap_mode="r")
        assert _LM[(level, f)].shape == (len(META[level]["pump"]), N_MELS, N_FRAMES)
    return _LM[(level, f)]


ALL_UNITS = [(lv, f, p) for lv in LEVELS for f in FRONTS for p in PUMPS]


def rows(level, p):
    m = META[level]
    return split_a(m["pump"], m["label"].astype(int), m["fname"], p)


def unit_file(lv, f, p):
    return OUT / "scores" / f"{lv}_{f}_{p}.npz"


def load_unit(lv, f, p):
    u = unit_file(lv, f, p)
    return dict(np.load(u, allow_pickle=False)) if u.exists() else {}


def save_unit(lv, f, p, d):
    u = unit_file(lv, f, p); tmp = u.with_name(u.stem + ".part.npz")
    np.savez(tmp, **d); os.replace(tmp, u)


def fit_fast(lv, f, p):
    # families 1, 2 and 3: seconds per unit
    tr, te = rows(lv, p)
    A, B = np.asarray(LM(lv, f)[tr], dtype=np.float32), np.asarray(LM(lv, f)[te], dtype=np.float32)
    la, lb = A.mean((1, 2)), B.mean((1, 2))                           # one number per clip: its overall loudness
    SA, SB = clip_stats(A), clip_stats(B)                             # 128 numbers per clip: the average and the wobble of each band
    return {"name": META[lv]["fname"][te], "label": META[lv]["label"][te].astype(np.int8), "F1": np.abs(lb - la.mean()) / max(float(la.std()), 1e-9),
            "F1_raw": lb, "F2": Maha(0.95).fit(SA).score(SB), "F3": IFfam().fit(SA).score(SB), "n_train": np.array(len(tr))}


def fit_ae(lv, f, p):
    tr, te = rows(lv, p)                                              # the rows in the order of the pump runs
    return AEfam().fit(np.asarray(LM(lv, f)[tr])).score(np.asarray(LM(lv, f)[te]))


t_fast = time.time()
for u in ALL_UNITS:
    d = load_unit(*u)
    if "F3" not in d:
        save_unit(*u, {**d, **fit_fast(*u)})
FAST_S = round(time.time() - t_fast, 1)
print(f"families 1, 2 and 3: {len(ALL_UNITS)} units fitted and scored · {FAST_S / 60:.1f} min")

In [ ]:
# 6 · The numbers: AUC of each pump, and the average of the four pumps, for every level, front end and family
def table(fams):
    res = {}
    for lv in LEVELS:
        for f in FRONTS:
            for fam in fams:
                per = {}
                for p in PUMPS:
                    d = load_unit(lv, f, p)
                    per[p] = round(float(roc_auc_score(d["label"], d[fam])), 4)
                per["avg"] = round(float(np.mean([per[p] for p in PUMPS])), 4)
                res[f"{lv}|{f}|{fam}"] = per
    return res


def show(res, fams):
    for fam in fams:
        print(f"\n{FAM_NAME[fam]} · average AUC of {len(PUMPS)} pumps")
        print(f"  {'level':7s}" + "".join(f"{FRONT_NAME[f]:>30s}" for f in FRONTS))
        for lv in LEVELS:
            print(f"  {lv:7s}" + "".join(f"{res[f'{lv}|{f}|{fam}']['avg']:30.4f}" for f in FRONTS))


FAST = ["F1", "F2", "F3", "F1_raw"]
RES_FAST = table(FAST); show(RES_FAST, FAST)
print("\n(family 4, the autoencoder, comes next: it is the slow part)")

In [ ]:
# 7 · Family 4, the autoencoder: one network per unit (the slow part). Each finished unit is saved in Drive › P04_ArraySimple › scores,
# so a session that stops is continued by Runtime › Run all
t_ae, done_now = time.time(), 0
todo = [u for u in ALL_UNITS if "F4" not in load_unit(*u)]
print(f"autoencoders to train: {len(todo)} of {len(ALL_UNITS)} units")
for k, u in enumerate(todo):
    t1 = time.time(); d = load_unit(*u); d["F4"] = fit_ae(*u); d["F4_seconds"] = np.array(round(time.time() - t1, 1))
    save_unit(*u, d); done_now += 1
    left = (time.time() - t_ae) / done_now * (len(todo) - done_now)
    print(f"  {k + 1}/{len(todo)} · {u[0]} {u[1]} {u[2]}: {time.time() - t1:.0f} s · about {left / 60:.0f} min left", flush=True)
AE_S = round(sum(float(load_unit(*u)["F4_seconds"]) for u in ALL_UNITS), 1)
print(f"family 4: all {len(ALL_UNITS)} units done · training time in total {AE_S / 60:.0f} min")

In [ ]:
# 8 · All four families, the gates, and the three questions of the rules
RES = table(SCORED); show(RES, SCORED)
print("\nFamily 5 · the pre-trained model alone (read from notebook 11's record, not run here)")
for lv in N11_FAMILY5_FROZEN:
    print(f"  {lv:7s}" + "".join(f"{N11_FAMILY5_FROZEN[lv][f]:30.4f}" for f in FRONTS))

# G0a · the delays measured here at 6 dB equal notebook 11's
TAU = json.loads((OUT / f"delays_{FIRST}.json").read_text())
G0A = {p: round(float(np.max(np.abs(np.array(TAU["tau"][p]) - np.array(N11_TAU_6DB[p])))), 4) for p in PUMPS if p in N11_TAU_6DB}
G0a = bool(G0A) and max(G0A.values()) <= 0.0625
print("\nG0a · delays at 6 dB against notebook 11, largest difference per pump (samples):", G0A, "→", "pass" if G0a else "FAIL")
# G0b · one microphone at 6 dB reproduces the pump runs
G0B, G0b = {}, True
for fam, tol in (("F2", 0.002), ("F3", 0.002), ("F1_raw", 0.002), ("F4", 0.02)):
    diff = {p: round(RES[f"{FIRST}|CH0|{fam}"][p] - PUMP_RECORD[fam][p], 4) for p in PUMPS if p in PUMP_RECORD[fam]}
    ok = bool(diff) and max(abs(v) for v in diff.values()) <= tol; G0B[fam] = {"difference": diff, "tolerance": tol, "pass": ok}; G0b &= ok
    print(f"G0b · {fam}: this notebook minus the pump runs, per pump {diff} (tolerance {tol}) →", "pass" if ok else "FAIL")
if not TEST:
    assert G0a and all(G0B[f]["pass"] for f in ("F2", "F3", "F1_raw")), "stop: the pipeline does not reproduce the records"

QUIET, LOUD = list(LEVELS)[0], list(LEVELS)[-1]
ANSWERS = {}
print(f"\nThe three questions, at {LOUD} (the loudest noise):")
for fam in FAMS:
    a = {f: RES[f"{LOUD}|{f}|{fam}"]["avg"] for f in FRONTS}; quiet = RES[f"{QUIET}|CH0|{fam}"]["avg"]
    hurts = quiet - a["CH0"] >= BAR_NOISE
    q1, q2 = a["BF"] - a["CH0"] >= BAR_Q1, a["BF"] - a["AVG"] >= BAR_Q2
    ANSWERS[fam] = {"one_microphone_quiet": quiet, **{f"{f}_loud": a[f] for f in FRONTS}, "noise_hurts": bool(hurts), "decidable": bool(hurts),
                    "Q1_aimed_array_helps": bool(q1), "Q2_aiming_not_only_more_microphones": bool(q2),
                    "BF_minus_CH0": round(a["BF"] - a["CH0"], 4), "BF_minus_AVG": round(a["BF"] - a["AVG"], 4)}
    said = (f"Q1 the aimed array helps: {'yes' if q1 else 'no'} ({a['BF'] - a['CH0']:+.4f}) · Q2 it is the aiming: {'yes' if q2 else 'no'} ({a['BF'] - a['AVG']:+.4f})"
            if hurts else "not decidable: the noise does not hurt this family on one microphone")
    print(f"  {fam}: one microphone {quiet:.4f} at {QUIET} → {a['CH0']:.4f} at {LOUD} · average of eight {a['AVG']:.4f} · aimed {a['BF']:.4f} · {said}")

In [ ]:
# 9 · Save, one picture, and one zip to download
RESULT = {"rules": RULES, "date": time.strftime("%Y-%m-%d %H:%M"), "test_run": TEST, "levels": list(LEVELS), "fronts": FRONTS, "pumps": PUMPS,
          "numpy": np.__version__, "sklearn": sklearn.__version__, "results": RES, "family5_frozen_notebook11": N11_FAMILY5_FROZEN,
          "delays_6dB": TAU, "G0a": {"largest_difference": G0A, "pass": G0a}, "G0b": G0B, "answers": ANSWERS,
          "n_train": {p: int(load_unit(FIRST, "CH0", p)["n_train"]) for p in PUMPS}, "n_test": {p: int(len(load_unit(FIRST, "CH0", p)["label"])) for p in PUMPS},
          "seconds": {"levels": {lv: float(META[lv]["seconds"]) for lv in LEVELS}, "families_1_2_3": FAST_S, "family_4": AE_S,
                      "family_4_per_unit": {"|".join(u): float(load_unit(*u)["F4_seconds"]) for u in ALL_UNITS}}}
(OUT / "array_simple.json").write_text(json.dumps(RESULT, indent=1, default=float, ensure_ascii=False))

import matplotlib
if not IN_COLAB:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
COL = {"CH0": "#8d8d8d", "AVG": "#eb6834", "BF": "#2a78d6"}
xs = list(range(len(LEVELS)))
fig, axes = plt.subplots(1, 5, figsize=(17, 3.9), sharey=True)
for ax, fam in zip(axes, FAMS + ["F5"]):
    for f in FRONTS:
        ys = [RES[f"{lv}|{f}|{fam}"]["avg"] for lv in LEVELS] if fam != "F5" else [N11_FAMILY5_FROZEN[lv][f] for lv in N11_FAMILY5_FROZEN]
        ax.plot(xs[:len(ys)], ys, "-o", color=COL[f], label=FRONT_NAME[f], lw=2)
    ax.axhline(0.5, color=MUTED, lw=0.8, ls="--")
    ax.set_xticks(xs, [lv.replace("dB", " dB").replace("-", "−") for lv in LEVELS])
    ax.set_title(FAM_NAME[fam].split(" (")[0] if fam != "F5" else "Family 5 · pre-trained model alone (notebook 11)", loc="left", color=INK, fontsize=9.5)
    ax.set_xlabel("factory noise grows →", color=MUTED); ax.grid(True, color=GRID, lw=0.6); ax.tick_params(colors=MUTED)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
axes[0].set_ylabel(f"average AUC over {len(PUMPS)} pumps", color=MUTED); axes[0].legend(frameon=False, loc="lower left", fontsize=8.5)
fig.tight_layout(); fig.savefig(OUT / "array_simple.png", dpi=150); plt.show()

ZIP = OUT / "p04_n34_results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for n in ("array_simple.json", "array_simple.png", f"delays_{FIRST}.json"):
        z.write(OUT / n, n)
    for u in ALL_UNITS:
        z.write(unit_file(*u), "scores/" + unit_file(*u).name)
print("saved array_simple.json · zipped →", ZIP, f"({ZIP.stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files as _f
    _f.download(str(ZIP))

## Send back
Send the zip that downloaded (`p04_n34_results.zip`). It is also in Drive › `P04_ArraySimple`. The large `lm_*.npy` files in that folder can be deleted once the zip is safe.